In [9]:
from pathlib import Path
import torch
import clip
from PIL import Image
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

In [10]:
# 1) Where is this notebook?
notebook_dir = Path.cwd()  
print("Notebook cwd:", notebook_dir)

# 2) Repo root is one level up
repo_root = notebook_dir.parent  
print("Repo root:", repo_root)

# 3) Now include the 'Versuch_2' folder in your data paths
normal_dir = (
    repo_root
    / "Versuch_2"
    / "Erdbeeren_yolo"
    / "Riseholme"
    / "Riseholme-2021"
    / "Data"
    / "Normal"
    / "Ripe"
)
anomaly_dir = (
    repo_root
    / "Versuch_2"
    / "Erdbeeren_yolo"
    / "Riseholme"
    / "Riseholme-2021"
    / "Data"
    / "Anomalous"
)

# 4) Sanity‐check that Python can see these folders
print("Normal dir exists?  ", normal_dir.exists())
print("Anomaly dir exists? ", anomaly_dir.exists())

# 5) Now glob your images
extensions = ("*.jpg", "*.jpeg", "*.png")
paths = sorted(str(p) for ext in extensions for p in normal_dir.glob(ext))
anomaly_paths = sorted(str(p) for ext in extensions for p in anomaly_dir.glob(ext))

print(f"Found {len(paths)} normal images\nFound {len(anomaly_paths)} anomaly images")

Notebook cwd: /home/parallels/Documents/Forschsem/Erdbeeren/CLIP
Repo root: /home/parallels/Documents/Forschsem/Erdbeeren
Normal dir exists?   True
Anomaly dir exists?  True
Found 462 normal images
Found 153 anomaly images


## 2. Split normals into train & validation

In [11]:
train_paths, val_normal_paths = train_test_split(
    paths, test_size=0.20, random_state=42, shuffle=True
)
print(f"{len(train_paths)} train normals, {len(val_normal_paths)} val normals")



369 train normals, 93 val normals


## 3. Load CLIP model

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

## 4. Feature extraction helper

In [13]:
def extract_embeddings(image_paths, model, preprocess, device, batch_size=4):
    all_feats = []
    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i : i + batch_size]
        imgs = torch.cat([
            preprocess(Image.open(p).convert("RGB")).unsqueeze(0)
            for p in batch
        ], dim=0).to(device)
        with torch.no_grad():
            feats = model.encode_image(imgs)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_feats.append(feats.cpu().numpy())
    return np.vstack(all_feats)

## 5. Train Isolation Forest on train normals

In [14]:
train_embeddings = extract_embeddings(train_paths, model, preprocess, device)
iso_forest = IsolationForest(contamination=0.01, random_state=0)
iso_forest.fit(train_embeddings)
print("Isolation Forest trained on normal embeddings")


Isolation Forest trained on normal embeddings


## 6. Calibrate threshold on validation normals

In [15]:
from sklearn.metrics import precision_recall_curve

# a) Berechne Scores auf Val-Normals und Anomalien
val_embeddings  = extract_embeddings(val_normal_paths, model, preprocess, device)
val_scores      = iso_forest.decision_function(val_embeddings)
anom_embeddings = extract_embeddings(anomaly_paths, model, preprocess, device)
anom_scores     = iso_forest.decision_function(anom_embeddings)

# b) Baue Label- und Score-Arrays (invertiert, damit hoch = anomal)
y_true      = np.concatenate([np.zeros_like(val_scores), np.ones_like(anom_scores)])
y_score_inv = -np.concatenate([val_scores, anom_scores])

# c) Precision–Recall-Kurve und F1 für jede Schwelle
precisions, recalls, thresholds = precision_recall_curve(y_true, y_score_inv)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

# d) Wähle die Schwelle mit maximalem F1
best_idx         = np.nanargmax(f1_scores)
best_threshold   = thresholds[best_idx]
best_precision   = precisions[best_idx]
best_recall      = recalls[best_idx]
best_f1          = f1_scores[best_idx]

print(f"Bestes F1 = {best_f1:.3f} bei Schwelle (invertiert) = {best_threshold:.4f}")
print(f"→ Precision = {best_precision:.3f}, Recall = {best_recall:.3f}")


Bestes F1 = 0.868 bei Schwelle (invertiert) = -0.0673
→ Precision = 0.854, Recall = 0.882


## 7. Evaluate on anomalies

In [16]:
# a) ROC-AUC (invertierte Scores)
roc_auc = roc_auc_score(y_true, y_score_inv)
print(f"ROC AUC: {roc_auc:.4f}")

# b) PR-AUC (Average Precision)
pr_auc = average_precision_score(y_true, y_score_inv)
print(f"PR AUC:  {pr_auc:.4f}")

# c) Konfusionsmatrix beim besten F1-Schwellenwert
#    (Weil y_score_inv hoch = stärker anomal, gilt pred=1 wenn y_score_inv >= best_threshold)
y_pred = (y_score_inv >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print(f"\nConfusion Matrix @ thr={best_threshold:.4f} → TN={tn}, FP={fp}, FN={fn}, TP={tp}\n")
print(classification_report(y_true, y_pred, target_names=["Normal","Anomalous"]))


ROC AUC: 0.9016
PR AUC:  0.9365

Confusion Matrix @ thr=-0.0673 → TN=70, FP=23, FN=18, TP=135

              precision    recall  f1-score   support

      Normal       0.80      0.75      0.77        93
   Anomalous       0.85      0.88      0.87       153

    accuracy                           0.83       246
   macro avg       0.82      0.82      0.82       246
weighted avg       0.83      0.83      0.83       246

